# Rotary Position Embedding

源码导航：[`compute_rope_freqs`](../../../core/position/rope.py#L17)、[`apply_rope`](../../../core/position/rope.py#L32)、[`RotaryPositionalEmbedding`](../../../core/position/rope.py#L46)。

RoPE 不把位置向量加到 token embedding，而是在 attention 内部旋转 Q/K 的二维配对。频率向量为：

$$
\omega_i=\theta^{-2i/d},\quad i=0,1,\ldots,d/2-1
$$

位置 $t$ 上的二维旋转可写成：

$$
\begin{bmatrix}x_1'\\x_2'\end{bmatrix}=\begin{bmatrix}\cos(t\omega)&-\sin(t\omega)\\\sin(t\omega)&\cos(t\omega)\end{bmatrix}\begin{bmatrix}x_1\\x_2\end{bmatrix}
$$

这样 Q/K 点积天然带有相对位置信息。Walkie 的改进思路是用 RoPE 替换 GPT-2 的 learned absolute position embedding，为长上下文和代码结构建模留出更好的外推空间。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.position.rope import RotaryPositionalEmbedding, apply_rope, compute_rope_freqs

## 1. 频率向量与缓存

In [ ]:
freqs = compute_rope_freqs(head_dim=16, base=1e6)
print(freqs.shape)
print(freqs[:5])

rope = RotaryPositionalEmbedding(head_dim=16, max_seq_len=8, base=1e6)
cos, sin = rope(seq_len=12, dtype=torch.float32)
print(cos.shape, sin.shape)

## 2. 旋转保持二维配对范数

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 4, 12, 16)  # B, H, T, D
cos, sin = rope(seq_len=12, dtype=x.dtype)
y = apply_rope(x, cos, sin)

half = x.size(-1) // 2
before = x[..., :half].pow(2) + x[..., half:].pow(2)
after = y[..., :half].pow(2) + y[..., half:].pow(2)
print('shape:', tuple(y.shape))
print('max norm diff:', (before - after).abs().max().item())

---

## 延伸阅读与参考资料

### 核心论文
- **RoFormer: Enhanced Transformer with Rotary Position Embedding**: Su et al., 2021. [arXiv:2104.09864](https://arxiv.org/abs/2104.09864)
- **LLaMA**: Touvron et al., 2023. [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)
- **Code Llama**: Roziere et al., 2023. [arXiv:2308.12950](https://arxiv.org/abs/2308.12950)

### 工程实现与博客
- **Hugging Face RoPE utilities**: [source](https://github.com/huggingface/transformers/blob/main/src/transformers/modeling_rope_utils.py)
- **RoPE 原理推导**: [kexue.fm](https://kexue.fm/archives/8265)